In [ ]:
"""
BEA Capital Data Extraction Pipeline
=====================================
Constructs asset-level rental prices (user costs) for IT / non-IT capital
aggregates following the Hall-Jorgenson framework, for use as inputs to a
nested CES production function.

All business logic lives in src/ (extractors/, parsers/, features/, schemas/,
pipeline.py) — this notebook is a thin narrative on top of it: configure,
call the pipeline, inspect and plot results. See doc-python.md for the
module layout, and src/jobs/build_ces_capital_inputs.py for the equivalent
standalone CLI job.

Key design decisions vs. naive implementation:
  - Restricted to NONRESIDENTIAL assets only: residential structures/equipment
    are excluded from both the capital stock and the r_t identity, making the
    stock denominator consistent with corporate net operating surplus (Pi_t),
    which excludes imputed rent on owner-occupied housing.
  - Internal rate of return r_t solved from the capital income exhaustion
    identity each year — not estimated from the CES, not taken from a market
    rate. This is pre-model arithmetic, keeping the CES identified.
  - Time-varying effective depreciation delta^s_t aggregated from detailed
    Hulten-Wykoff rates via rental-share-weighted Tornqvist — enters the
    aggregate accumulation equation as data, not as a calibrated constant.
  - Investment price deflator pi_{j,t} smoothed with a 3-year centred MA
    before entering the rental price formula — mandatory for IT assets where
    raw year-over-year changes can be extreme.

API key setup:
    BEA_API_KEY is read from the .env file (gitignored) via src.config.settings.
    Register a free key at: https://apps.bea.gov/api/signup/

Outputs (written to settings.paths.outputs by the CLI job — this notebook
keeps everything in-memory as `result`, a pipeline.CapitalPipelineResult):
    ces_capital_inputs.csv     — K_IT, K_non_IT, p_IT, p_non_IT,
                                  delta_IT, delta_non_IT, r_t  (annual)
    ces_data_panel.csv         — full CES panel incl. output-value aggregate
    asset_dimension_table.csv  — static asset seed with delta_j, bucket,
                                  residential flag
    diagnostic_checks.csv      — r_t sanity, delta implied vs constructed,
                                  Pi_t vs sum_stock ratio

Notation reference:
    j          — detailed BEA asset type (line number in FAA Table 2.1)
    s          — capital bucket: "IT" or "non_IT"
    t          — year
    V_{j,t}    — current-cost net stock of asset j at t   [Table 2.1]
    Q^N_{j,t}  — chain-quantity index for net stock       [Table 2.4]
    X_{j,t}    — current-cost investment in asset j       [Table 2.5]
    Z_{j,t}    — chain-quantity index for investment      [Table 2.6]
    p^I_{j,t}  — investment goods price deflator (derived from 2.5 / 2.6)
    K^N_{j,t}  — real net stock (derived from 2.1 + 2.4)
    pi_{j,t}   — year-over-year growth in p^I_{j,t}  (raw)
    pi_bar     — 3-year centred MA of pi_{j,t}        (smoothed)
    delta_j    — geometric depreciation rate (BEA/Hulten-Wykoff, static)
    Pi_t       — net operating surplus, corporate business [NIPA T11400 L8]
    r_t        — internal rate of return (solved from exhaustion identity)
    p^K_{j,t}  — rental price / user cost of asset j
    omega_{j,t}— rental income share of asset j within its bucket
    K^s_t      — Tornqvist capital services index for bucket s
    p^s_t      — dual rental price index for bucket s
    delta^s_t  — effective (rental-share-weighted) depreciation for bucket s
"""

In [ ]:
from src.config import sources
from src.config.settings import settings
from src.extractors.bea_api import discover_table_names
from src.pipelines.bea_pipeline import run_capital_pipeline
from src.utils.logging import configure_logging

configure_logging()
beakey = settings.bea_api_key

# Configuration

In [ ]:
# All constants (REF_YEAR, YEAR_START, YEAR_END, YEARS, PI_SMOOTH_WINDOW,
# PI_FLOOR, FA_TABLE_*, NIPA_TABLE_*, NOS_LINE_*, VA_TABLE_*, EFF_START)
# live in src/config/sources.py.
print(
    f"Sample: {sources.YEAR_START}–{sources.YEAR_END}  |  ref year: {sources.REF_YEAR}  |  pi window: {sources.PI_SMOOTH_WINDOW}"
)

# Run the pipeline

Everything from BEA table fetch through the Hall-Jorgenson capital-services
math to the final CES panel happens inside `run_capital_pipeline` — see
`src/pipeline.py`. This is the same function `src/jobs/build_ces_capital_inputs.py`
calls to produce the CLI job's output CSVs.

In [ ]:
result = run_capital_pipeline(beakey)

# Inspect results

`result` is a `pipeline.CapitalPipelineResult` — the fields below are the same
tables the CLI job writes to CSV, kept in memory here for interactive
inspection. Structured progress/diagnostic logs from the run above (r_t
decomposition, sanity checks, decade-mean rental-price decomposition) came
from `src.features.diagnostics`, via structlog.

In [ ]:
print("Nonresidential coverage of total private stock (recent years):")
print(result.nonres_coverage.tail(5).round(3))
print("Expected: ~0.55–0.65")

print("\nNOS scope ratio (nonfarm-nonfin-private / corporate), recent years:")
print(result.scope_ratio.tail(5).round(3))

In [ ]:
print("CES input table (last 5 years):")
result.ces_inputs.tail().round(4)

In [ ]:
print("Full CES data panel (last 5 years):")
result.ces_data.tail().round(4)

In [ ]:
print("Diagnostics table (last 5 years):")
result.diagnostics.tail().round(4)

# Exploratory: BEA table discovery (optional)

Run interactively when you need to find/verify a BEA table name or line
number — not part of the pipeline itself (doc-python.md: extractors don't
bake ad-hoc inspection into the pipeline; do it here instead).

In [ ]:
discover_table_names(beakey, "FixedAssets", "net stock")

# Plots

In [ ]:
result.ces_data[["p_IT_real", "p_non_IT_real"]].plot(
    title="Rental price indices (IT vs non-IT)"
)

In [ ]:
result.ces_data.plot(y="r_t_real", title="Internal Rate of Return (real)")